In [17]:
# Phase 9: Modeling — Customer Churn Prediction

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# -----------------------------
# Setup paths
# -----------------------------
project_dir = "/content/drive/MyDrive/Cape Stone Project 2026"
features_path = f"{project_dir}/data/processed/features.csv"
models_dir = f"{project_dir}/models"
viz_dir = f"{project_dir}/visualizations"

os.makedirs(models_dir, exist_ok=True)
os.makedirs(viz_dir, exist_ok=True)

# -----------------------------
# Load features safely
# -----------------------------
if not os.path.exists(features_path):
    raise FileNotFoundError(f"❌ Features file not found at {features_path}. Run Phase 8 first to generate it.")

features = pd.read_csv(features_path)
print("✅ Features loaded successfully")
print("Shape:", features.shape)
print("Columns:", features.columns.tolist())

# -----------------------------
# Prepare data
# -----------------------------
numeric_features = features.select_dtypes(include=["number"]).copy()
X = numeric_features.drop("churn_flag", axis=1)
y = numeric_features["churn_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# Model 1: Logistic Regression
# -----------------------------
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)
print("\nLogistic Regression Performance:")
print(classification_report(y_test, y_pred_lr))

# -----------------------------
# Model 2: Gradient Boosting
# -----------------------------
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
print("\nGradient Boosting Performance:")
print(classification_report(y_test, y_pred_gb))

# -----------------------------
# Save best model + metadata
# -----------------------------
best_model = gb
joblib.dump(best_model, os.path.join(models_dir, "final_model.pkl"))

metrics = {
    "logistic_regression": {
        "accuracy": accuracy_score(y_test, y_pred_lr),
        "precision": precision_score(y_test, y_pred_lr),
        "recall": recall_score(y_test, y_pred_lr),
        "f1": f1_score(y_test, y_pred_lr)
    },
    "gradient_boosting": {
        "accuracy": accuracy_score(y_test, y_pred_gb),
        "precision": precision_score(y_test, y_pred_gb),
        "recall": recall_score(y_test, y_pred_gb),
        "f1": f1_score(y_test, y_pred_gb)
    }
}

with open(os.path.join(models_dir, "model_metadata.json"), "w") as f:
    json.dump(metrics, f, indent=4)

print("\n✅ Best model saved at:", os.path.join(models_dir, "final_model.pkl"))
print("✅ Metrics saved at:", os.path.join(models_dir, "model_metadata.json"))

# -----------------------------
# Visualizations
# -----------------------------
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_gb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix — Gradient Boosting")
plt.savefig(f"{viz_dir}/confusion_matrix.png")
plt.close()

# Feature Importance
importances = gb.feature_importances_
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)

importance_df.plot(kind="barh", x="feature", y="importance", legend=False, color="skyblue")
plt.title("Feature Importance — Gradient Boosting")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(f"{viz_dir}/feature_importance.png")
plt.close()

# Correlation Heatmap
plt.figure(figsize=(8,6))
sns.heatmap(numeric_features.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.savefig(f"{viz_dir}/correlation_heatmap.png")
plt.close()

# Recency vs Churn Distribution
plt.figure(figsize=(8,6))
sns.histplot(data=features, x="recency", hue="churn_flag", bins=30, kde=True, palette="Set1")
plt.title("Distribution of Recency by Churn Flag")
plt.savefig(f"{viz_dir}/distributions.png")
plt.close()

print("✅ All plots generated and saved in /visualizations folder")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Features loaded successfully
Shape: (5922, 21)
Columns: ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'gender', 'date_of_birth', 'signup_date', 'state', 'city', 'membership_tier', 'preferred_channel', 'marketing_opt_in', 'referral_source', 'recency', 'frequency', 'monetary', 'ticket_count', 'avg_satisfaction', 'avg_resolution_time', 'churn_flag']

Logistic Regression Performance:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       632
         1.0       1.00      1.00      1.00       553

    accuracy                           1.00      1185
   macro avg       1.00      1.00      1.00      1185
weighted avg       1.00      1.00      1.00      1185


Gradient Boosting Performance:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       632
   